In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_69_try_4.pickle

In [ ]:
%%cudf.pandas.profile
### cell 70 ###
def grab_subset_of_data_82(original_df, question_of_interest):
    # 1) pick columns via GPU string contains
    col_mask = original_df.columns.to_series().str.contains(question_of_interest)
    subset_df = original_df.loc[:, col_mask]

    # 2) GPU‐accelerated split/get/lstrip on a Series
    new_cols = (
        subset_df.columns
                 .to_series()
                 .str.split("-")
                 .str.get(-1)
                 .str.lstrip()
    )
    # bring back to Python list once, then assign
    subset_df.columns = new_cols.tolist()

    # 3) drop rows where all are null via GPU boolean indexing
    valid = ~subset_df.isna().all(axis=1)
    return subset_df.loc[valid]

question_of_interest_cell82 = 'Who/what are your favorite media sources that report on data science topics?'
media_df_2022 = grab_subset_of_data_82(responses_df_2022_cell10, question_of_interest_cell82)
media_df_2022.info()